In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

# ==========================================
# 1. LỚP XỬ LÝ DỮ LIỆU (Dành cho Ngọc)
# ==========================================
class HouseDataPipeline:
    def __init__(self, csv_path, json_path):
        self.csv_path = csv_path
        self.json_path = json_path
        self.df = None
        self.zone_dict = {}

    def load_and_merge_data(self):
        try:
            # Đọc file CSV
            self.df = pd.read_csv(self.csv_path)
            
            # Đọc file JSON
            with open(self.json_path, 'r') as f:
                self.zone_dict = json.load(f)
                
            # Tạo cột mới từ việc map JSON vào data
            self.df['Zone_Name'] = self.df['MSZoning'].map(self.zone_dict)
            print(f"Đã tải thành công! Kích thước dữ liệu: {self.df.shape}")
        except FileNotFoundError as e:
            print(f"LỖI: Không tìm thấy file dữ liệu. Chi tiết: {e}")

    def clean_data(self):
        print("Đang làm sạch dữ liệu...")
        num_cols = self.df.select_dtypes(include=['int64', 'float64']).columns
        cat_cols = self.df.select_dtypes(include=['object']).columns
        
        # Điền giá trị thiếu (missing values)
        self.df[num_cols] = self.df[num_cols].fillna(self.df[num_cols].median())
        self.df[cat_cols] = self.df[cat_cols].fillna(self.df[cat_cols].mode().iloc[0])
        
        # Xóa dữ liệu trùng lặp (duplication)
        self.df = self.df.drop_duplicates()
        
        # Cắt bỏ ngoại lai (outliers) có diện tích lớn bất thường (GrLivArea > 4000)
        self.df = self.df[self.df['GrLivArea'] < 4000]
        
        print("-> Đã làm sạch xong!")

    def perform_eda(self):
        print("Đang vẽ 5 biểu đồ EDA...")
        plt.figure(figsize=(18, 12))
        
        # 1. Phân phối Giá nhà
        plt.subplot(2, 3, 1)
        sns.histplot(self.df['SalePrice'], kde=True, color='blue')
        plt.title('1. Phân phối Giá nhà (SalePrice)')
        
        # 2. Diện tích vs Giá nhà
        plt.subplot(2, 3, 2)
        sns.scatterplot(x='GrLivArea', y='SalePrice', data=self.df, color='orange')
        plt.title('2. Diện tích vs Giá nhà')
        
        # 3. Chất lượng nhà vs Giá nhà
        plt.subplot(2, 3, 3)
        sns.boxplot(x='OverallQual', y='SalePrice', data=self.df, palette='viridis')
        plt.title('3. Chất lượng nhà vs Giá nhà')
        
        # 4. Số lượng nhà theo MSZoning
        plt.subplot(2, 3, 4)
        sns.countplot(x='MSZoning', data=self.df, palette='Set2')
        plt.title('4. Số lượng nhà theo MSZoning')
        
        # 5. Ma trận tương quan
        plt.subplot(2, 3, 5)
        num_df = self.df.select_dtypes(include=['int64', 'float64'])
        top_corr = num_df.corr().nlargest(10, 'SalePrice')['SalePrice'].index
        sns.heatmap(self.df[top_corr].corr(), annot=True, cmap='coolwarm', fmt=".2f")
        plt.title('5. Ma trận tương quan')
        
        plt.tight_layout()
        plt.show()

# ==========================================
# 2. LỚP MÔ HÌNH HỌC MÁY (Dành cho Trâm)
# ==========================================
class HousePriceModel:
    def __init__(self, df):
        self.df = df
        self.model = None

    def preprocess_for_ml(self):
        print("Đang chuẩn bị dữ liệu cho Machine Learning...")
        pass

    def train_model(self):
        print("Đang huấn luyện mô hình...")
        pass

# ==========================================
# 3. HÀM MAIN - KÍCH HOẠT CHẠY TOÀN BỘ (Rất quan trọng)
# ==========================================
if __name__ == "__main__":
    print("--- BẮT ĐẦU CHẠY PIPELINE ---")
    
    pipeline = HouseDataPipeline(csv_path="./data/train.csv", json_path="./data/zone_info.json")
    pipeline.load_and_merge_data()
    pipeline.clean_data()
    pipeline.perform_eda()
    
    if pipeline.df is not None:
        ml_system = HousePriceModel(pipeline.df)
        ml_system.preprocess_for_ml()
        ml_system.train_model()